# PyTorch 零基础 6/6：Dataset、DataLoader 与变长语音 Batch

这是第 1～41 课 ASR 主线之前的桥梁课。先预测，再运行；看懂输出后必须改一个值验证自己的解释。

| 项目 | 内容 |
|---|---|
| 前置要求 | 完成基础 5；会定义 nn.Module 和训练步骤 |
| 建议投入 | 60～90 分钟，可分两次完成 |
| 核心概念 | Dataset 与 DataLoader、collate、padding 与 lengths、mask 与有效区域 |
| 完成标准 | 能解释代码、独立完成练习、从空白重写本课核心函数 |


## 课前诊断（先不要运行代码）

1. 用自己的话解释：Dataset 与 DataLoader。
2. 猜测 collate、padding 与 lengths 最容易出现哪一种错误。
3. 写下你对 mask 与有效区域 的暂时理解；不会可以明确写“不知道”。

这三题不计分，只用于留下学习前证据。


## 1. Dataset 定义一个样本，DataLoader 组织迭代与 batch


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


class ToySpeechDataset(Dataset):
    def __init__(self):
        self.waveforms = [
            torch.tensor([0.1, 0.2, 0.3]),
            torch.tensor([0.4, 0.5, 0.6, 0.7, 0.8]),
            torch.tensor([0.9, 1.0]),
        ]
        self.labels = [0, 1, 0]

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, index):
        return self.waveforms[index], self.labels[index]


dataset = ToySpeechDataset()
waveform, label = dataset[1]
print(len(dataset), waveform.shape, label)
assert len(dataset) == 3
assert waveform.shape == (5,)


## 2. 变长语音不能直接 stack，需要 collate


In [ ]:
def speech_collate(batch):
    waveforms, labels = zip(*batch)
    lengths = torch.tensor([waveform.numel() for waveform in waveforms], dtype=torch.long)
    padded = pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    labels = torch.tensor(labels, dtype=torch.long)
    return {"waveforms": padded, "lengths": lengths, "labels": labels}


loader = DataLoader(dataset, batch_size=3, shuffle=False, collate_fn=speech_collate)
batch = next(iter(loader))
for key, value in batch.items():
    print(key, value.shape, value.dtype)

assert batch["waveforms"].shape == (3, 5)
assert torch.equal(batch["lengths"], torch.tensor([3, 5, 2]))
assert batch["labels"].shape == (3,)


## 3. lengths 生成 mask，区分真实数据与 padding


In [ ]:
def lengths_to_mask(lengths: torch.Tensor, max_length: int | None = None) -> torch.Tensor:
    if lengths.ndim != 1:
        raise ValueError(f"expected [B] lengths, got {tuple(lengths.shape)}")
    if (lengths < 0).any():
        raise ValueError("lengths must be non-negative")
    if max_length is None:
        max_length = int(lengths.max()) if lengths.numel() else 0
    steps = torch.arange(max_length, device=lengths.device)
    return steps.unsqueeze(0) < lengths.unsqueeze(1)


mask = lengths_to_mask(batch["lengths"], batch["waveforms"].shape[1])
print(mask)
assert mask.shape == (3, 5)
assert torch.equal(mask.sum(dim=1), batch["lengths"])


## 4. Masked mean 不让 padding 污染统计量


In [ ]:
padded = batch["waveforms"]
valid_sum = (padded * mask).sum(dim=1)
masked_mean = valid_sum / batch["lengths"].clamp_min(1)

manual = torch.tensor([
    dataset.waveforms[0].mean(),
    dataset.waveforms[1].mean(),
    dataset.waveforms[2].mean(),
])
print("masked mean:", masked_mean)
assert torch.allclose(masked_mean, manual)


## 5. 每个 batch 都要审计接口契约


In [ ]:
def audit_speech_batch(batch):
    waveforms = batch["waveforms"]
    lengths = batch["lengths"]
    labels = batch["labels"]
    assert waveforms.ndim == 2
    assert lengths.shape == labels.shape == (waveforms.shape[0],)
    assert waveforms.dtype == torch.float32
    assert lengths.dtype == labels.dtype == torch.long
    assert (lengths <= waveforms.shape[1]).all()
    assert torch.isfinite(waveforms).all()


audit_speech_batch(batch)
print("batch contract passed")


## 本课练习（保留作答区）


1. 用一句话区分 Dataset 与 DataLoader。
2. 为什么变长波形不能直接 `torch.stack`？
3. 写出示例 batch 中 waveforms、lengths、labels 的 shape。
4. 根据 lengths=[3,5,2] 手画 bool mask。
5. 解释为什么只补零但不传 length/mask 会出错。
6. 为 collate 加入空波形策略并说明选择。
7. 为 lengths_to_mask 写正常、全零、空、负数测试。
8. 故意把 labels 变成 float，确认审计器能发现。
9. 比较普通 mean 和 masked mean 的数值差异。
10. 画出 Dataset -> collate -> padded batch -> encoder -> loss 的数据流并标 shape/dtype。


评分：每题 0～2 分。达到 16/20 可以继续；12～15 分次日重做错题；低于 12 分回看代码并从空白复现。


## 离场票与间隔复习

- [ ] 我能闭卷解释：Dataset 与 DataLoader、collate、padding 与 lengths、mask 与有效区域。
- [ ] 我能预测核心代码的 shape、dtype 或数值方向。
- [ ] 我能从空白重写至少一个函数，并通过正常、边界、错误输入测试。
- [ ] 我能说出一个“代码能运行但语义错误”的例子。

复习安排：明天闭卷回忆 5 分钟；7 天后重做第 4、7、10 题；30 天后重新构造最小实验。

下一步：进入 ASR 主线第 1 课《声音与采样》，之后第 7～9 课会把这些能力用于声学编码器。
